# AlphaZeroSAT
*Learning a SAT branching heuristic in the style of Alpha(Go) Zero (self-play + MCTS).*

End-to-end, reproducible pipeline (setup &rarr; data &rarr; training &rarr; evaluation &rarr; results) for the
search-based model of the [NeuroSAT](https://github.com/dmeoli/NeuroSAT) project. The implementation lives in
the `.py` modules; this notebook only orchestrates the steps.

## 1. Introduction
We treat SAT solving as a **game**: each move is a branching literal, and a **CNN** with a policy head
($\pi$) and a value head ($v$) guides a **Monte Carlo Tree Search** (MCTS). The network is trained by
**self-play** with the AlphaZero loss (cross-entropy on the MCTS policy + MSE on the outcome + L2). The state
is the CNF as a fixed-size *clauses &times; variables* matrix &mdash; simple, but it **cannot generalise across
sizes** (the motivation for the graph models in the companion notebook).

Metric (Wang &amp; Rompf, 2018): the **mean number of branching decisions** to solve a problem (lower is better).

## 2. Setup

### 2.1 GPU, Google Drive and repository

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > GPU'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)
    CKPT_DIR = '/content/gdrive/MyDrive/neuroSAT_ckpts'
except Exception as e:
    print('Drive mount failed (%s) -> local checkpoints' % e)
    CKPT_DIR = '/content/neuroSAT_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True); print('checkpoints ->', CKPT_DIR)

if not os.path.isdir('/content/neuroSAT/.git'):
    !git clone --recurse-submodules https://github.com/dmeoli/NeuroSAT /content/neuroSAT
%cd /content/neuroSAT
!git pull origin master 2>&1 | tail -1
!git submodule sync --recursive >/dev/null 2>&1; git submodule update --init --recursive --force 2>&1 | tail -2

### 2.2 Install dependencies

In [ ]:
!pip -q install gymnasium PyYAML numpy
!sudo apt-get -qq install -y swig zlib1g-dev >/dev/null
import sys; PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
!sudo apt-get -qq install -y {PYV}-dev >/dev/null
print('build prereqs ready for', PYV)

### 2.3 Build the native MCTS MiniSat environment (GSL-free)

In [ ]:
import os
%cd /content/neuroSAT/AlphaZeroSAT
!PYTHON=python3 bash MCTSminisat/build_so.sh 2>&1 | tail -4
print('AlphaZeroSAT env built:', os.path.exists('MCTSminisat/minisat/gym/_GymSolver.so'))

## 3. Preprocessing
The model trains on **uf20-91** (uniform-random 3-SAT, 20 variables / 91 clauses), already split into
train/test in the shared `data/` hub &mdash; no further preprocessing is needed (the CNF is read directly into
the fixed-size matrix by the env).

In [ ]:
%cd /content/neuroSAT
!echo "train: $(ls data/uf20-91/train_v0 | wc -l) cnf | test: $(ls data/uf20-91/test_v0 | wc -l) cnf"

## 4. Model
**State**: the CNF as a fixed-size *clauses &times; variables* matrix (entries $\{+1,0,-1\}$ by polarity).
**Network**: a CNN trunk followed by a **policy** head $\pi$ over the $2n$ literal actions (invalid literals
masked) and a **value** head $v\in[-1,1]$. **MCTS** uses $(\pi,v)$ to run simulations (PUCT selection,
expand, back-up) and returns an improved policy from the visit counts; **self-play** generates the training
targets. (Full architecture & math: see the slides / report.) Implementation: `models_torch.py` (Model1/2/3),
`alphazero_torch.py` (the AlphaZero loss + trainer), `mct.py` (MCTS).

## 5. Training
Self-play + supervised training alternate for a number of **cycles**; `train_torch.py` checkpoints to Drive
and logs the validation metric each cycle.

In [ ]:
%cd /content/neuroSAT/AlphaZeroSAT
import os; os.makedirs(f'{CKPT_DIR}/alphazero', exist_ok=True)
!python3 train_torch.py \
    --train_path ../data/uf20-91/train_v0 --eval_path ../data/uf20-91/test_v0 --device auto \
    --n_batch 16 --n_repeat 10 --resign 400 --cycles 25 --sl_num_steps 1000 --sl_n_batch 32 --lr 1e-3 \
    --save_path {CKPT_DIR}/alphazero/az_model.pt 2>&1 | tee -a {CKPT_DIR}/alphazero/az_train.log

## 6. Evaluation
Measure the paper's metric &mdash; the mean number of branching decisions to solve the held-out test set
(lower is better) &mdash; for the trained model.

In [ ]:
%cd /content/neuroSAT/AlphaZeroSAT
!python3 eval_torch.py --model_path {CKPT_DIR}/alphazero/az_model.pt --eval_path ../data/uf20-91/test_v0 --n_files 64

## 7. Results
The per-cycle validation curve is in the training log (`az_train.log`); the final mean-decisions value above is
the headline number. Compare it against a random policy / MiniSat as reported in the slides and the report.

In [ ]:
%cd /content/neuroSAT/AlphaZeroSAT
import re, statistics
vals = [float(m.group(1)) for l in open(f'{CKPT_DIR}/alphazero/az_train.log', errors='ignore')
        for m in [re.search(r'eval mean decisions ([\d.]+)', l)] if m]
if vals:
    print(f'cycles logged: {len(vals)} | last-5 mean: {statistics.fmean(vals[-5:]):.2f} | best: {min(vals):.2f}')

## 8. Conclusions
- AlphaZeroSAT learns a usable branching heuristic from self-play, with no human data.
- Its **fixed-size CNN cannot generalise across problem sizes** &mdash; the key limitation that the graph-based
  models (Graph-Q-SAT / GAT-Q-SAT, companion notebook) overcome with a size-agnostic graph representation.